# litert

> Google's [litert_lm](https://github.com/google-ai-edge/litert-lm) backend for `.litertlm` Gemma builds — litert message helpers, GPU/NPU backends, KV-cache usage, and `bench()`.

Tools, streaming, HITL, PyFence, grading, and skill install live in [core](core.html). This notebook covers litert wire format and `LitertChat`.


In [ ]:
#| default_exp litert

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, re, os, asyncio, io, base64, uuid, warnings
from html import escape
from mimetypes import guess_type
from contextlib import ExitStack, redirect_stdout
from litert_lm import (Engine, Backend, ConstrainedDecodingConfig, Conversation, Session, Message, Contents, Content, Role, ToolCall,
                       ToolEventHandler, SamplerConfig, Benchmark, set_min_log_severity)
from litert_lm._messages import Text, ImageBytes, ImageFile, AudioBytes, AudioFile, ToolResponse, normalize_message
from huggingface_hub import hf_hub_download, list_repo_files, scan_cache_dir
from fastcore.all import Path, store_attr, patch, L, GetAttr, ifnone, detect_mime, first, listify, img_bytes, AttrDict, in_, str2bool
from safepyrun import RunPython
from rishi import core
from rishi.core import *


/Users/71293/code/personal/orgs/ramabana/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Messages

litert's `Message` / `Content` types — rishi wraps them with `_mk_content`, `_mk_msg`, `_mk_msgs` for the shared history format.


In [ ]:
#| export
def _mk_content(o):
    'Convert `o` to a litert `Content`, sniffing bytes/files for image vs audio.'
    if isinstance(o, Content): return o
    if isinstance(o, str): return Text(o)
    if isinstance(o, bytes): return AudioBytes(o) if (detect_mime(o) or '').startswith('audio/') else ImageBytes(o)
    if isinstance(o, Path): return AudioFile(str(o)) if (guess_type(str(o))[0] or '').startswith('audio/') else ImageFile(str(o))
    raise TypeError(f"Unsupported content type: {type(o)}")

def _mk_msg(content, role='user'):
    'Create a litert `Message` from str/bytes/list/dict/Message.'
    if content is None: return None
    if isinstance(content, Message): return content
    if isinstance(content, dict): return Message(Role(content['role']), Contents.of(content['content']))
    parts = [_mk_content(o) for o in content] if isinstance(content, list) else [_mk_content(content)]
    return Message(Role(role), Contents(parts))

def _mk_msgs(msgs):
    'Normalize a list of messages to litert message dicts.'
    if not msgs: return []
    return [normalize_message(m if isinstance(m, (Message, dict)) else _mk_msg(m)) for m in listify(msgs)]

In [ ]:
from fastcore.test import test_eq, test_fail

In [ ]:
test_eq(_mk_msg("hello").to_json(), {"role": "user", "content": [{"type": "text", "text": "hello"}]})
test_eq(_mk_msg("hi", role="model").to_json()["role"], "model")
test_eq([x["role"] for x in _mk_msgs(["a", _mk_msg("b", role="model")])], ["user", "model"])
assert isinstance(_mk_content("x"), Text)
# bytes are sniffed: image vs audio
assert isinstance(_mk_content(b'\x89PNG\r\n\x1a\n' + b'\x00'*16), ImageBytes)
assert isinstance(_mk_content(b'RIFF\x00\x00\x00\x00WAVE' + b'\x00'*16), AudioBytes)

## Portable history

`fmt2hist` / `hist2fmt` convert litert messages to the canonical rishi history dicts (and back), so a conversation can hop to llama.cpp or a hosted model via `messages=`.


In [ ]:
#| export
#: Short unique id for litert tool-call messages.
def _call_id(): return f"call_{uuid.uuid4().hex[:8]}"

_ph = {'image': '[image]', 'audio': '[audio]'}
def _canon_content(c):
    "litert content (str or list of parts) -> a canonical string; media -> `[image]`/`[audio]`."
    if c is None or isinstance(c, str): return c
    return '\n'.join(p.get('text', '') if p.get('type') == 'text' else _ph.get(p.get('type'), '[media]')
                     for p in c if isinstance(p, dict))

def _fmt2hist(m, dflt_role='user'):
    "One litert message (str/`Message`/native dict/`Resp`) -> a canonical rishi history dict."
    if isinstance(m, str): return {'role': 'user', 'content': m}
    if isinstance(m, Message): m = m.to_json()
    elif not isinstance(m, dict): m = _mk_msg(m).to_json()
    role = 'assistant' if m.get('role') == 'model' else m.get('role', dflt_role)
    content = m.get('content')
    if role == 'tool':
        if isinstance(content, list):
            tr = first(content, lambda p: isinstance(p, dict) and p.get('type') == 'tool_response') or {}
            out = {'role': 'tool', 'name': tr.get('name', ''), 'content': str(tr.get('response', ''))}
        else: out = {'role': 'tool', 'content': content if isinstance(content, str) else str(content)}
        if m.get('name'): out['name'] = m['name']
        if m.get('tool_call_id'): out['tool_call_id'] = m['tool_call_id']
        return out
    out = {'role': role}
    if (cc := _canon_content(content)) is not None: out['content'] = cc
    if m.get('channels'): out['channels'] = dict(m['channels'])
    if m.get('tool_calls'):
        out['tool_calls'] = [{'id': tc.get('id') or _call_id(), 'type': 'function',
                              'function': {'name': tc.get('function', {}).get('name', ''),
                                           'arguments': tc.get('function', {}).get('arguments', {})}}
                             for tc in m['tool_calls']]
        out.setdefault('content', '')
    return out

def _data_uri_bytes(url):
    "Decode a `data:...;base64,...` URI to bytes, else None."
    if isinstance(url, str) and url.startswith('data:') and ';base64,' in url:
        return base64.b64decode(url.split(';base64,', 1)[1])
    return None

def _litert_contents(c):
    "Canonical content (str or OpenAI-style parts) -> a litert `Contents`."
    if c is None: return Contents.empty()
    if isinstance(c, str): return Contents.of(c)
    parts = []
    for p in c:
        if not isinstance(p, dict): continue
        t = p.get('type')
        if t == 'text': parts.append(Text(p.get('text', '')))
        elif t == 'image_url':
            u = p.get('image_url', {}); url = u.get('url') if isinstance(u, dict) else u
            parts.append(ImageBytes(b) if (b := _data_uri_bytes(url)) is not None else ImageFile(url))
        elif t == 'input_audio': parts.append(AudioBytes(base64.b64decode((p.get('input_audio') or {}).get('data', ''))))
    return Contents(parts)

def _hist2fmt1(m):
    "One canonical rishi history dict (or raw input) -> a litert `Message` for the engine."
    if isinstance(m, Message): return m
    if isinstance(m, str): return Message.user(m)
    if not isinstance(m, dict): return _mk_msg(m)
    role = m.get('role', 'user')
    if role == 'tool': return Message.tool(Contents([ToolResponse(m.get('name', ''), m.get('content', ''))]))
    tcs = [ToolCall(tc.get('function', {}).get('name', ''), tc.get('function', {}).get('arguments', {}))
           for tc in (m.get('tool_calls') or [])]
    return Message(Role('model' if role == 'assistant' else role), _litert_contents(m.get('content')),
                   tool_calls=tcs, channels=(m.get('channels') or None))

def _to_litert_msg(m):
    "Raw input or a (canonical/native) chat dict -> a litert `Message` for the engine preface."
    return m if isinstance(m, Message) else (_hist2fmt1(m) if isinstance(m, dict) else _mk_msg(m))



## Litert callbacks

Three litert-specific callbacks plus `ChatToolHandler`, which bridges litert's native tool API to rishi's approval gate. Token counts come from `conv.token_count` after each turn.


In [ ]:
#| export
class _ToolReminderCallback(ChatCallback):
    'Inject a tool-summary reminder into the outgoing message when tools are registered.'
    order = 30
    def __init__(self, tool_reminder=tool_reminder_): store_attr()
    def before_send(self):
        if self.chat.tools and self.chat.turn_msg is not None: self.chat.turn_msg.contents.contents.append(Text(self.tool_reminder))


In [ ]:
#| export
class _HistoryCallback(ChatCallback):
    'Record the outgoing message and the response into `chat.hist` (canonical form).'
    order = 0
    def before_send(self):
        if self.chat.turn_msg is not None:
            self.chat.hist.append(_fmt2hist(self.chat.turn_msg))
            self.chat._turn_msg_recorded = True
    def after_response(self): self.chat.hist.append(_fmt2hist(self.chat.turn_res, 'assistant'))

class _UsageCallback(ChatCallback):
    'Fold each response\'s token usage into `chat.use` from a `token_count` diff.'
    order = 10
    def after_response(self):
        c = self.chat; delta = c.conv.token_count - c._tc0
        out = len(c.engine.tokenize(resp_text(c.turn_res)))
        new_prompt, cached = max(delta - out, 0), c._tc0
        c.use += UsageStats(prompt_tokens=cached + new_prompt, completion_tokens=out,
                            total_tokens=cached + delta, n=1, cached_tokens=cached)


## Tool calling

`ChatToolHandler` records tool rounds in litert's wire format. Approval runs through the shared `approve` hook — see [core](core.html#human-in-the-loop-tool-approval).


In [ ]:
#| export
#: Tool name from an OpenAI-shaped tool-call dict.
def _tc_name(tc): return tc.get('function', {}).get('name', '')

class ChatToolHandler(ToolEventHandler):
    "Bridge litert's in-engine tool loop to Chat callbacks, HITL approval, the tool-call budget, and history."
    def __init__(self, chat): self.chat = chat
    def approve_tool_call(self, tool_call):
        c = self.chat
        c.turn_tc = tool_call
        for _ in run_cbs(c, 'before_tool_calls'): pass
        over = c._budget_exceeded or (c.max_steps is not None and c._steps >= c.max_steps)
        ok = False if over else (c.approve(tool_call) if c.approve else True)
        if ok: c._steps += 1
        else: c._budget_exceeded = c._budget_exceeded or over
        fn = tool_call.get('function', {}); self._tcid = _call_id()
        c.hist.append({'role': 'assistant', 'content': '',
            'tool_calls': [{'id': self._tcid, 'type': 'function',
                            'function': {'name': fn.get('name', ''), 'arguments': fn.get('arguments', {})}}]})
        if not ok: c.hist.append({'role': 'tool', 'tool_call_id': self._tcid, 'name': fn.get('name', ''),
                                  'content': budget_msg_ if over else 'Denied by human operator'})
        return ok
    def process_tool_response(self, tool_response):
        mx = getattr(self.chat, 'tool_max_len', None)
        if mx and isinstance(tool_response, str) and len(tool_response) > mx:
            tool_response = tool_response[:mx] + ' …[truncated]'
        self.chat.turn_tool_result = tool_response
        self.chat.hist.append({'role': 'tool', 'tool_call_id': getattr(self, '_tcid', None),
                               'name': _tc_name(self.chat.turn_tc), 'content': str(tool_response)})
        for _ in run_cbs(self.chat, 'after_tool_calls'): pass
        return tool_response


In [ ]:
#| hide
from fastcore.test import test_eq

class _FakeChat:
    "Just the attributes `ChatToolHandler` touches - enough to drive it without an engine."
    def __init__(self, approve=None, max_steps=None):
        self.hist, self.cbs, self.approve, self.tool_max_len = [], L(), approve, None
        self.max_steps, self._steps, self._budget_exceeded = max_steps, 0, False

tc = {'function': {'name': 'add', 'arguments': {'a': 1, 'b': 2}}}

ch = _FakeChat(); h = ChatToolHandler(ch)
test_eq(h.approve_tool_call(tc), True)
test_eq(ch.hist[-1]['role'], 'assistant')
test_eq(h.process_tool_response('3'), '3')
test_eq(ch.hist[-1]['role'], 'tool'); test_eq(ch.hist[-1]['content'], '3')

# HITL: `approve` can deny, and the model is told
ch = _FakeChat(approve=lambda tc: False); h = ChatToolHandler(ch)
test_eq(h.approve_tool_call(tc), False)
test_eq(ch.hist[-1]['content'], 'Denied by human operator')

# budget: past `max_steps` calls, further ones are denied whatever `approve` says
ch = _FakeChat(max_steps=1); h = ChatToolHandler(ch)
test_eq(h.approve_tool_call(tc), True)
test_eq(h.approve_tool_call(tc), False)
test_eq(ch.hist[-1]['content'], budget_msg_)
assert ch._budget_exceeded          # `Chat.__call__` now sends `final_prompt` to close the turn out

# max_steps=None means no cap
ch = _FakeChat(); h = ChatToolHandler(ch)
assert all(h.approve_tool_call(tc) for _ in range(20))

## Loading models & LitertChat

`_get_model` downloads or cache-hits a `.litertlm` build. Pass `backend=Backend.GPU()` (or set `RISHI_LITERT_GPU=1`) for GPU/NPU; CPU is the default. `LitertChat.create_engine` owns the litert `Conversation`.


In [ ]:
#| export
#: Default litert-community model ids.
gemma4_e4b='litert-community/gemma-4-E4B-it-litert-lm'
gemma4_e2b='litert-community/gemma-4-E2B-it-litert-lm'
gemma4_12b='litert-community/gemma-4-12B-it-litert-lm'


In [ ]:
#| export
#: `RISHI_LITERT_GPU=0` forces CPU when GPU/NPU init fails.
LITERT_GPU = str2bool(os.getenv('RISHI_LITERT_GPU', '1').lower())
def _litertlm(fs):
    "First native `.litertlm` path in `fs` (skips `-web`/other builds)."
    return first(fs, lambda p: p.endswith('.litertlm') and 'web' not in p)

def _cached_model(model_id):
    "Local `.litertlm` path from the HF cache without hitting the network, else None."
    try: repo = first(scan_cache_dir().repos, lambda r: r.repo_id == model_id)
    except Exception: return None
    return _litertlm(str(f.file_path) for r in repo.revisions for f in r.files) if repo else None

def _get_model(model_id, model_path=None):
    "Return a local `.litertlm` path: `model_path`, else HF cache, else download."
    if model_path and Path(model_path).exists(): return model_path
    if (hit := _cached_model(model_id)): return hit
    if not (fn := _litertlm(list_repo_files(model_id))): raise FileNotFoundError(f"No .litertlm file found for {model_id}")
    return hf_hub_download(model_id, fn)

def _merge_chunks(chunks):
    "Reconstruct an assistant response dict (text + thinking) from streamed litert chunks."
    text, th = ''.join(resp_text(c) for c in chunks), ''.join(thought(c) for c in chunks)
    r = {'role': 'assistant', 'content': [{'type': 'text', 'text': text}]}
    if th: r['channels'] = {'thought': th}
    return Resp(r)

_dflt_cbs = [_HistoryCallback, _UsageCallback, _ToolReminderCallback, SlidingWindowCallback]

class LitertChat(core.Chat):
    "Sync chat over a local litert_lm engine."
    _runtime = 'litert'
    _dflt_cbs = _dflt_cbs
    mk_content, mk_msg, mk_msgs = staticmethod(_mk_content), staticmethod(_mk_msg), staticmethod(_mk_msgs)

    @staticmethod
    def fmt2hist(msgs):
        "litert-native messages -> canonical rishi history dicts."
        return [_fmt2hist(m) for m in listify(msgs)]
    @staticmethod
    def hist2fmt(msgs):
        "Canonical rishi history dicts -> litert `Message`s for the engine."
        return [_hist2fmt1(m) for m in listify(msgs)]

    @classmethod
    def create_engine(cls, model_id=gemma4_e2b, model_path=None, be=None, vbe=None, abe=None,
                      multimodal=True, cache_dir=None, enable_speculative_decoding=None, **kw):
        'Build a litert `Engine` on the GPU unless told otherwise; creates `cache_dir` if given. Override/`@patch` to customize.'
        be, vbe, abe = kw.pop('backend', be), kw.pop('vision_backend', vbe), kw.pop('audio_backend', abe)
        if cache_dir: Path(cache_dir).mkdir(parents=True, exist_ok=True)
        mod = _get_model(model_id, model_path)
        def _mk(b):
            mm = dict(vision_backend=ifnone(vbe, Backend.CPU()), audio_backend=ifnone(abe, Backend.CPU()))
            return Engine(mod, backend=b, cache_dir=cache_dir or '',
                          enable_speculative_decoding=enable_speculative_decoding, **(mm if multimodal else {}), **kw)
        if be is not None: return _mk(be)
        if LITERT_GPU:
            try: return _mk(Backend.GPU())
            except Exception as e:
                warnings.warn(f'litert GPU backend unavailable ({type(e).__name__}: {e}) - falling back to the '
                              'CPU. Set RISHI_LITERT_GPU=0 to stop asking for it.')
        return _mk(Backend.CPU())

    def __init__(self, model=None, *, runtime=None, model_path=None, engine=None, backend=None,
                 multimodal=True, cache_dir=None, enable_speculative_decoding=None, eng_kw=None,
                 sp='', messages=None, tools=None, ctx_limit=None, approve=None, tool_max_len=None,
                 max_steps=10, final_prompt=dflt_final_prompt_, parallel_tools=False, max_parallel_tools=None,
                 think=False, filter_think=True, temp=None, top_k=None, top_p=None,
                 seed=None, sampler_config=None, max_output_tokens=None, constrain=None, conv_kw=None,
                 cbs=None, default_cbs=True):
        if parallel_tools: raise NotImplementedError(
            "litert runs its tool loop inside the engine, so it can't dispatch calls in parallel; "
            "use runtime='llama' (or 'mlx') for parallel_tools=True.")
        model = core.split_runtime(model)[1]
        model_id = None if model is None or core._is_path(model) else model
        model_path = model_path or (model if model and core._is_path(model) else None)
        self._stack, self._conv_stack = ExitStack(), ExitStack()
        self._own_engine = engine is None
        if self._own_engine:
            engine = self.create_engine(model_id or gemma4_e2b, model_path, backend,
                multimodal=multimodal, cache_dir=cache_dir, enable_speculative_decoding=enable_speculative_decoding, **(eng_kw or {}))
            self.engine = self._stack.enter_context(engine)
        else: self.engine = engine
        # the system prompt is kept apart from `hist` and re-applied on every rebuild, so eviction can
        # never drop it - it is the anchor a sliding window is supposed to preserve
        self._sys_pre = [{'role': 'system', 'content': sp}] if sp else []
        if sampler_config is None and any(x is not None for x in (temp, top_k, top_p, seed)):
            sampler_config = SamplerConfig(temperature=temp, top_k=top_k, top_p=top_p, seed=seed)
        cvk = dict(sampler_config=sampler_config, max_output_tokens=max_output_tokens, **(conv_kw or {}))
        if think: cvk['extra_context'] = {**cvk.get('extra_context', {}), 'enable_thinking': True}
        if filter_think: cvk['filter_channel_content_from_kv_cache'] = True
        if (legacy := cvk.pop('enable_constrained_decoding', None)) is not None:
            warnings.warn("enable_constrained_decoding was litert's own name for this and is "
                          'not accepted any more; pass constrain=True/False to rishi instead.',
                          DeprecationWarning, stacklevel=2)
            if constrain is None: constrain = bool(legacy)
        self._constrain = constrain
        self._conv_kw, self.tools, self.conv = cvk, L(tools), None
        self.tool_handler = ChatToolHandler(self)
        self._mk_conv(self._sys_pre + [_to_litert_msg(m) for m in listify(messages)])
        self.ctx_limit = ctx_limit
        self._setup(model=model, sp=sp, messages=messages, tools=tools, approve=approve,
                    tool_max_len=tool_max_len, max_steps=max_steps, max_parallel_tools=max_parallel_tools,
                    final_prompt=final_prompt, cbs=cbs, default_cbs=default_cbs)

    def _constrained(self):
        "The `constrained_decoding_config` to build with, or `None`. Default is on when there are tools."
        if 'constrained_decoding_config' in self._conv_kw: return None
        want = bool(self.tools) if self._constrain is None else bool(self._constrain)
        return ConstrainedDecodingConfig(enable=True) if want else None

    def _mk_conv(self, messages=None):
        "Build the conversation from `messages`, releasing any current one first."
        self._conv_stack.close()
        self._conv_stack = ExitStack()
        kw = dict(self._conv_kw)
        if (cd := self._constrained()) is not None: kw['constrained_decoding_config'] = cd
        self.conv = self._conv_stack.enter_context(self.engine.create_conversation(
            messages=messages or None, tools=list(self.tools) or None,
            tool_event_handler=self.tool_handler, **kw))
        return self.conv

    def _set_sp(self, sp):
        "The system prompt lives outside `hist` here, so `reconfigure` has to replace it explicitly."
        self._sys_pre = [{'role': 'system', 'content': sp}] if sp else []

    def _recreate_conv(self):
        "Rebuild the `Conversation` from the current (possibly evicted) `hist`, re-applying the system prompt."
        self._mk_conv(self._sys_pre + [_to_litert_msg(m) for m in self.hist])

    def recover_context(self, err, max_output_tokens=None):
        "Recover a full context by evicting the middle of history and retrying the current message."
        return self._retry_evicted(err, max_output_tokens)

    def _retry_evicted(self, err, max_output_tokens=None, keep_first=2, keep_last=6):
        "The window filled up mid-turn: evict the middle of `hist`, rebuild, and send the same turn again."
        kept, dropped = evict_middle(self.hist, keep_first, keep_last)
        if not dropped:
            raise ContextWindowExceededError(f"context window full and nothing left to evict: {err}") from err
        self.hist[:] = kept
        self.evicted = getattr(self, 'evicted', 0) + len(dropped)
        # The outgoing message may already be in Python history, but it is about to be
        # sent again. Do not put that copy in the rebuilt engine preface.
        prior = self.hist[:-1] if self._turn_msg_recorded else self.hist
        self._mk_conv(self._sys_pre + [_to_litert_msg(m) for m in prior])
        self._tc0 = self.conv.token_count            # the rebuilt cache is a new baseline for usage
        try: return self.conv.send_message(self.turn_msg, max_output_tokens=max_output_tokens)
        except RuntimeError as e:
            raise ContextWindowExceededError(
                f"could not recover after evicting {len(dropped)} messages: {err}") from err

    @property
    def token_count(self): return self.conv.token_count
    @property
    def cached_tokens(self):
        "Tokens already resident in this conversation's KV cache and reusable by the next turn."
        return self.conv.token_count
    def cancel(self):
        "Cancel the in-flight generation."
        self.conv.cancel_process()
    def count_tokens(self, text):
        "Number of tokens in `text` per the engine tokenizer."
        return len(self.engine.tokenize(text))
    def render(self, msg):
        "The exact templated string litert will send for `msg`."
        return self.conv.render_message_to_string(_mk_msg(msg))
    def _send(self, msg, max_output_tokens=None):
        'Send one message through the callback pipeline, evicting and retrying once if the context window fills up.'
        self.turn_msg = _mk_msg(msg)
        self._turn_msg_recorded = False
        for _ in run_cbs(self, 'before_send'): pass
        self._tc0 = self.conv.token_count      # after the callbacks: one of them may have rebuilt `conv`
        try: r = self.conv.send_message(self.turn_msg, max_output_tokens=max_output_tokens)
        except RuntimeError as e:
            if not is_ctx_error(self, e): raise
            r = self.recover_context(e, max_output_tokens)
        self.turn_res = Resp(r)
        for _ in run_cbs(self, 'after_response'): pass
        return self.turn_res
    def _stream(self, msg, max_output_tokens=None, cbs=None):
        'Stream a turn as markdown chunks; per-call `cbs` live only for this turn.'
        added = self.add_cbs(cbs); prev = getattr(self, '_streaming', False); self._streaming = True
        try:
            self.turn_msg = _mk_msg(msg)
            for _ in run_cbs(self, 'before_send'): pass
            self._tc0 = self.conv.token_count   # after the callbacks: one of them may have rebuilt `conv`
            fmt, chunks = StreamFormatter(), []
            for o in self.conv.send_message_async(self.turn_msg, max_output_tokens=max_output_tokens):
                chunks.append(o); yield self._emit(o, fmt)
            self.turn_res = _merge_chunks(chunks)
            yield from run_cbs(self, 'after_response')
            return self.turn_res   # stream's final Resp, captured by SaveReturn / AsyncChat `.value`
        finally: self._streaming = prev; self.remove_cbs(added)
    def _oneshot(self, prompt, sp='', think=None, max_tokens=None):
        """Stateless one-shot completion text via a throwaway conversation."""
        kw = {'extra_context': {'enable_thinking': True}} if think else {}
        if max_tokens is not None: kw['max_output_tokens'] = int(max_tokens)
        pre = [{'role': 'system', 'content': sp}] if sp else None
        with self.engine.create_conversation(messages=pre, **kw) as conv:
            return resp_text(conv.send_message(prompt))
    def _structured_call(self, prompt, schema, sp):
        "Forced tool call; falls back to parsing a JSON reply when the model answers in prose."
        pre = [{'role': 'system', 'content': sp}] if sp else None
        with self.engine.create_conversation(messages=pre, tools=[schema], automatic_tool_calling=False) as conv:
            r = conv.send_message(prompt)
        if tcs := r.get('tool_calls'): return tcs[0].get('function', {}).get('arguments', {})
        try: return json.loads(extract_fence(resp_text(r), 'json'))
        except (json.JSONDecodeError, TypeError) as e: raise ValueError(f"model neither called the tool nor returned JSON; reply: {resp_text(r)[:200]!r}") from e
    def close(self):
        "Release this chat's conversation and, only when owned, its engine (idempotent)."
        if getattr(self, '_conv_stack', None) is not None: self._conv_stack.close(); self._conv_stack = None
        if getattr(self, '_own_engine', False) and getattr(self, '_stack', None) is not None:
            self._stack.close()
        self._stack = None
        self.conv = None


In [ ]:
#| hide
# which accelerator gets asked for is decided before any weights are touched, so a fake `Engine`
# is enough to test it - and the GPU is what a caller who named nothing now gets
_asked, _real_engine, _real_get, _real_gpu = [], Engine, _get_model, LITERT_GPU
def _fake_engine(mod, backend=None, **kw): _asked.append(type(backend).__name__); return 'engine'
try:
    Engine, _get_model = _fake_engine, lambda *a, **kw: 'model'
    LitertChat.create_engine()
    LitertChat.create_engine(be=Backend.CPU())                  # named positionally
    LitertChat.create_engine(**{'backend': Backend.CPU()})      # ...or through `eng_kw`, as litert spells it
    LITERT_GPU = False
    LitertChat.create_engine()
    test_eq(_asked, ['GPU', 'CPU', 'CPU', 'CPU'])

    # a build whose GPU delegate is missing warns and gets the CPU rather than taking the model down
    LITERT_GPU, _asked[:] = True, []
    def _no_gpu(mod, backend=None, **kw):
        if type(backend).__name__ == 'GPU': raise RuntimeError('no GPU delegate in this build')
        _asked.append('CPU'); return 'engine'
    Engine = _no_gpu
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter('always')
        test_eq(LitertChat.create_engine(), 'engine')
    test_eq(_asked, ['CPU'])
    assert 'RISHI_LITERT_GPU=0' in str(w[0].message)
finally: Engine, _get_model, LITERT_GPU = _real_engine, _real_get, _real_gpu

In [ ]:
#| hide
# The accelerator, named the way `eng_kw` names it. A recording stand-in for `Engine` keeps this
# model-free: what matters is which `backend` the engine is handed, not that one loads.
class _RecEngine:
    def __init__(self, *a, **kw): _RecEngine.kw = kw

_cpu, _gpu = type(Backend.CPU()), type(Backend.GPU())
_Engine, _gm = Engine, _get_model
Engine, _get_model = _RecEngine, lambda *a, **kw: 'model.litertlm'
try:
    LitertChat.create_engine('m', be=Backend.CPU())
    test_eq(type(_RecEngine.kw['backend']), _cpu)
    LitertChat.create_engine('m', backend=Backend.GPU())                        # what `eng_kw` carries
    test_eq(type(_RecEngine.kw['backend']), _gpu)
    LitertChat.create_engine('m', None, Backend.CPU(), backend=Backend.GPU())   # named beats positional
    test_eq(type(_RecEngine.kw['backend']), _gpu)
    LitertChat.create_engine('m', vision_backend=Backend.GPU(), audio_backend=Backend.GPU())
    test_eq((type(_RecEngine.kw['vision_backend']), type(_RecEngine.kw['audio_backend'])), (_gpu, _gpu))
finally: Engine, _get_model = _Engine, _gm


In [ ]:
#| hide
# rebuilding the conversation: a fake engine lets us check what litert would be handed
class _FakeConv:
    def __init__(self, messages=None, **kw): self.messages, self.token_count, self.sent, self.kw = list(messages or []), 0, [], kw
    def __enter__(self): return self
    def __exit__(self, *a): self.closed = True
    def send_message(self, m, **kw): self.sent.append(m); return {'role': 'assistant', 'content': [{'type': 'text', 'text': 'ok'}]}

class _FakeEngine:
    def __init__(self): self.convs, self.exited = [], False
    def __enter__(self): return self
    def __exit__(self, *a): self.exited = True
    def create_conversation(self, messages=None, **kw):
        c = _FakeConv(messages, **kw); self.convs.append(c); return c
    def tokenize(self, t): return [0] * max(1, len(str(t)) // 4)

eng = _FakeEngine()
chat = LitertChat(engine=eng, sp='anchor me', ctx_limit=100)
test_eq(len(eng.convs), 1)
chat.hist += [{'role': 'user', 'content': f'q{i}'} for i in range(6)]
chat._recreate_conv()
test_eq(len(eng.convs), 2)
assert eng.convs[0].closed                              # the old conversation was released
test_eq(eng.convs[1].messages[0], {'role': 'system', 'content': 'anchor me'})   # sp re-applied
test_eq(len(eng.convs[1].messages), 7)                  # system prompt + the six history messages
chat.close()
assert not eng.exited                              # a caller-owned engine remains usable by sibling chats

# the reactive path: a context error evicts the middle and resends the same turn
class _FullConv(_FakeConv):
    def send_message(self, m, **kw):
        self.sent.append(m)
        if len(self.sent) == 1 and not getattr(self, 'ok', False): raise RuntimeError('max number of tokens reached')
        return {'role': 'assistant', 'content': [{'type': 'text', 'text': 'recovered'}]}

class _FullEngine(_FakeEngine):
    def create_conversation(self, messages=None, **kw):
        c = _FullConv(messages, **kw); c.ok = bool(self.convs); self.convs.append(c); return c

eng2 = _FullEngine()
chat2 = LitertChat(engine=eng2, sp='anchor', ctx_limit=100)
chat2.hist += [{'role': 'user', 'content': f'q{i}'} for i in range(20)]
r = chat2('one more')
test_eq(resp_text(r), 'recovered')
assert chat2.evicted > 0
assert len(eng2.convs) == 2                             # rebuilt exactly once
assert all(str(m) != 'one more' for m in eng2.convs[1].messages)  # resent, not prefixed twice
test_eq([m['content'] for m in chat2.hist][:2], ['q0', 'q1'])

# nothing left to evict -> a typed error rather than a raw litert traceback
eng3 = _FullEngine()
chat3 = LitertChat(engine=eng3, ctx_limit=100)
test_fail(lambda: chat3('hi'), contains='nothing left to evict')

In [ ]:
#| hide
# constrained decoding, decided where the tools are known
def _atool(x: int) -> int:
    "Double x."
    return x * 2

def _cd(**kw):
    e = _FakeEngine(); LitertChat(engine=e, **kw)
    return e.convs[0].kw.get('constrained_decoding_config')

test_eq(_cd(tools=[_atool]).enable, True)                  # on when there are tools
test_eq(_cd(), None)                                       # off when there are none
test_eq(_cd(constrain=True).enable, True)                  # ...and forced either way
test_eq(_cd(tools=[_atool], constrain=False), None)
mine = ConstrainedDecodingConfig(enable=False)
assert _cd(tools=[_atool], conv_kw=dict(constrained_decoding_config=mine)) is mine

# litert's old argument name, translated rather than passed on to raise
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    test_eq(_cd(conv_kw=dict(enable_constrained_decoding=True)).enable, True)
assert any(w.category is DeprecationWarning for w in caught)

# tools handed over later are covered, because the conversation is rebuilt
e = _FakeEngine(); ch = LitertChat(engine=e)
test_eq(e.convs[0].kw.get('constrained_decoding_config'), None)
ch.reconfigure(tools=[_atool])
test_eq(e.convs[-1].kw['constrained_decoding_config'].enable, True)

In [ ]:
# fmt2hist / hist2fmt round-trip (model-free)
_nat = [
    'hi',
    Message.model(Contents.of('hello')).to_json(),
    {'role': 'model', 'tool_calls': [{'type': 'function', 'function': {'name': 'add', 'arguments': {'a': 2, 'b': 3}}}]},
    {'role': 'tool', 'content': [{'type': 'tool_response', 'name': 'add', 'response': 5}]},
]
_canon = LitertChat.fmt2hist(_nat)
test_eq(_canon[0], {'role': 'user', 'content': 'hi'})
test_eq(_canon[1], {'role': 'assistant', 'content': 'hello'})
test_eq(_canon[2]['role'], 'assistant')
test_eq(_canon[2]['tool_calls'][0]['function'], {'name': 'add', 'arguments': {'a': 2, 'b': 3}})
assert _canon[2]['tool_calls'][0]['id']                       # a synthetic id was assigned
test_eq(_canon[3], {'role': 'tool', 'name': 'add', 'content': '5'})

# canonical -> litert Messages ready for the engine
_ms = LitertChat.hist2fmt(_canon)
test_eq([m.role.value for m in _ms], ['user', 'model', 'model', 'tool'])
test_eq(str(_ms[1]), 'hello')
test_eq(_ms[2].tool_calls[0].name, 'add')
test_eq(_ms[3].contents.to_json(), [{'type': 'tool_response', 'name': 'add', 'response': '5'}])

# fmt2hist is idempotent on an already-canonical (llama-origin) tool round
_llama = [{'role': 'assistant', 'content': '', 'tool_calls': [{'id': 'call_x', 'type': 'function',
                                                               'function': {'name': 'add', 'arguments': {'a': 1}}}]},
          {'role': 'tool', 'tool_call_id': 'call_x', 'name': 'add', 'content': '1'}]
test_eq(LitertChat.fmt2hist(_llama), _llama)

In [ ]:
#| eval: false
set_min_log_severity(5)
_get_model(gemma4_12b)

'/Users/71293/.cache/huggingface/hub/models--litert-community--gemma-4-12B-it-litert-lm/snapshots/44cf85a326f79b814fa86a60af414c042755b43a/gemma-4-12B-it.litertlm'

## Using Chat (manual)

GPU-backed demos below. First run downloads weights; set `cache_dir` to reuse them.


In [ ]:
#| eval: false
chat=Chat(backend=Backend.GPU(), cache_dir='.cache/litertlm', think=True)
# chat_12 = LitertChat(engine=LitertChat.create_engine(gemma4_12b, multimodal=False, cache_dir='.cache/litertlm', be=Backend.GPU()))

In [ ]:
#| eval: false
# set_min_log_severity(2)
r = chat("Reply with exactly: pong")
assert 'pong' in resp_text(r).lower()
assert resp_text(chat.hist[-1]) == resp_text(r) and chat.hist[0]['role'] == 'user'
assert chat.use.total_tokens > 0 and chat.token_count > 0
print(chat.use); chat.print_hist()

total=104|in=103|out=1|turns=1


**user**

Reply with exactly: pong

---

**assistant**

> **🧠 Thinking**
>
> Thinking Process:
> 
> 1.  **Analyze the Request:** The user has instructed me to reply with *exactly* the word "pong".
> 2.  **Determine the Constraint:** The constraint is strict: "Reply with exactly: pong".
> 3.  **Formulate the Response:** The response must be the string "pong".
> 4.  **Final Output Generation:** pong

pong

In [ ]:
#| eval: false
# Real LiteRT conversation-cache test. BenchmarkInfo reports the actual last prefill performed by
# the runtime; turn two should prefill only its short new message while retaining the old KV context.
kvchat = Chat(backend=Backend.GPU(), cache_dir='.cache/litertlm', think=False, max_output_tokens=16,
              eng_kw={'enable_benchmark': True})
try:
    kvchat(('cache verification context ' * 96) + '\nReply with exactly: stored')
    first = kvchat.conv.get_benchmark_info()
    context_before_turn2 = kvchat.token_count
    r = kvchat('What exact word did I ask you to reply with? Reply with only that word.')
    second = kvchat.conv.get_benchmark_info()
    print(dict(first_prefill=first.last_prefill_token_count,
               second_prefill=second.last_prefill_token_count,
               context_before_turn2=context_before_turn2,
               context_after_turn2=kvchat.token_count,
               reported_cached=kvchat.use.cached_tokens,
               answer=resp_text(r)))
    assert context_before_turn2 > 0 and kvchat.token_count > context_before_turn2
    assert second.last_prefill_token_count < context_before_turn2, 'turn two appears to have re-prefilled the old context'
    assert kvchat.use.cached_tokens == context_before_turn2, kvchat.use
    assert 'stored' in resp_text(r).lower(), resp_text(r)
finally:
    kvchat.close()


{'first_prefill': 304, 'second_prefill': 25, 'context_before_turn2': 306, 'context_after_turn2': 333, 'reported_cached': 306, 'answer': 'stored'}


### Images and audio

Gemma-4 litert builds accept images and audio beside text in one call — same message list shape as other backends.


In [ ]:
#| eval: false
from PIL import Image
from fastcore.all import img_bytes, Path
im = Image.open(repo_root()/'nbs'/'images.jpeg')
chat(['explain this image', img_bytes(im)])
chat(['transcribe this audio', AudioFile(str(repo_root()/'nbs'/'speech.wav'))])

> **🧠 Thinking**
>
> Thinking Process:
> 
> 1.  **Analyze the Request:** The user wants me to transcribe the provided audio.
> 2.  **Analyze the Input (Audio/Text):** The input is a piece of poetry or song lyrics.
> 3.  **Perform Transcription (Segment by Segment):** I will listen/read the provided text and transcribe it accurately.
> 
>     *   *Original text:* "Dancing in the masquerade, idle truth in plain sight jaded, pop, roll, click, dot, who will I be today or not. But such a tide as moving seems asleep, too full for sound and foam, when that drew from out the boundless deep turns again home, twilight and evening bell, and after that"
> 
> 4.  **Review and Finalize Transcription:** Ensure the transcription matches the provided text exactly. (The provided text is already quite clean, so the task is primarily verification and output.)
> 
> 5.  **Output Generation.**

Dancing in the masquerade, idle truth in plain sight jaded, pop, roll, click, dot, who will I be today or not. But such a tide as moving seems asleep, too full for sound and foam, when that drew from out the boundless deep turns again home, twilight and evening bell, and after that

In [ ]:
#| hide
merged = _merge_chunks([{"content": [{"type": "text", "text": "po"}]},
                        {"content": [{"type": "text", "text": "ng"}]}])
test_eq(merged, {"role": "assistant", "content": [{"type": "text", "text": "pong"}]})

## Utilities & bench

`classify`, `structured`, and `check` are inherited from core. **`bench()`** is litert-only: init time, time-to-first-token, prefill/decode tok/s. LiteRT does not expose log-likelihood scoring.


In [ ]:
#| export
def bench(model_id=gemma4_e2b, model_path=None, backend=Backend.CPU(), prefill_tokens=64, decode_tokens=64, **kw):
    "Benchmark init time, TTFT, and prefill/decode tokens-per-sec via litert's `Benchmark`."
    return Benchmark(_get_model(model_id, model_path), backend=backend, prefill_tokens=prefill_tokens, decode_tokens=decode_tokens, **kw).run()


In [ ]:
#| eval: false
# classify + structured on a real model
sm = Chat(cache_dir='.cache/litertlm')
test_eq(sm.classify("I absolutely loved this film!", ['positive', 'negative']), 'positive')

from dataclasses import dataclass
@dataclass
class Person: name:str; age:int
p = sm.structured("Extract the person: John Smith is 30 years old.", Person)
print(p); assert isinstance(p, Person) and 'John' in p.name
sm.close()

Person(name='John Smith', age=30)


In [ ]:
from fastcore.test import test_eq
import rishi.core, rishi.litert
test_eq(rishi.core.get_runtime('litert'), rishi.litert.LitertChat)

# dispatch really happens in `Chat.__new__`, so it can be checked without loading a model
test_eq(type(rishi.core.Chat.__new__(rishi.core.Chat)), rishi.litert.LitertChat)              # the default runtime
test_eq(type(rishi.core.Chat.__new__(rishi.core.Chat, 'litert-community/gemma-4-E2B-it-litert-lm')),
        rishi.litert.LitertChat)
test_eq(rishi.litert.LitertChat._runtime, 'litert')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

## Shared LiteRT engine broker

A LiteRT `Engine` is native process state, so separate harnesses cannot attach to it directly. `LitertBroker` owns one engine in a long-lived process and offers isolated conversations through a local Unix socket. Turns are serialized because LiteRT does not guarantee concurrent mutation of one engine.

In [ ]:
#| export
import socket, threading

def _send_json(sock, obj):
    data = json.dumps(obj).encode()
    sock.sendall(len(data).to_bytes(4, 'big') + data)

def _recv_json(sock):
    def read(n):
        out = bytearray()
        while len(out) < n:
            if not (part := sock.recv(n - len(out))): raise EOFError
            out.extend(part)
        return bytes(out)
    n = int.from_bytes(read(4), 'big')
    if n > 1_000_000: raise ValueError('broker request too large')
    return json.loads(read(n))

class LitertBroker:
    'Serve isolated string-only LiteRT conversations over a local Unix socket with one shared engine.'
    def __init__(self, address, engine=None, model_id=gemma4_e2b, **engine_kw):
        self.address, self.engine, self._own_engine = str(address), engine, engine is None
        self.model_id, self.engine_kw = model_id, engine_kw
        self.lock, self.state_lock, self.listener, self.thread = threading.Lock(), threading.Lock(), None, None
        self.clients, self.handlers, self.stopping = set(), set(), False

    def _free_socket(self):
        if not Path(self.address).exists(): return
        probe = socket.socket(socket.AF_UNIX)
        try:
            probe.connect(self.address)
        except ConnectionRefusedError:
            Path(self.address).unlink()
        else:
            raise RuntimeError(f'LiteRT broker already running at {self.address}')
        finally: probe.close()

    def start(self):
        with self.state_lock:
            if self.listener is not None: return self
            self._free_socket()
            listener = None
            try:
                if self.engine is None: self.engine = LitertChat.create_engine(self.model_id, **self.engine_kw)
                listener = socket.socket(socket.AF_UNIX)
                listener.bind(self.address); listener.listen()
            except Exception:
                if listener is not None: listener.close()
                if self._own_engine and hasattr(self.engine, 'close'): self.engine.close(); self.engine = None
                Path(self.address).unlink(missing_ok=True)
                raise
            self.stopping, self.listener = False, listener
            self.thread = threading.Thread(target=self._serve, args=(listener,), daemon=True)
            self.thread.start()
        return self

    def _serve(self, listener):
        while True:
            try: conn, _ = listener.accept()
            except OSError: break
            with self.state_lock:
                if self.stopping:
                    conn.close(); break
                self.clients.add(conn)
            handler = threading.Thread(target=self._handle, args=(conn,), daemon=True)
            with self.state_lock: self.handlers.add(handler)
            handler.start()

    def _handle(self, conn):
        conv = None
        try:
            while not self.stopping:
                req = _recv_json(conn)
                if not isinstance(req, dict) or req.get('op') not in {'open', 'send', 'close'}: raise ValueError('invalid broker request')
                op = req['op']
                if op == 'open':
                    if conv is not None or not isinstance(req.get('messages', []), list) or not all(isinstance(m, str) for m in req.get('messages', [])): raise ValueError('invalid conversation')
                    with self.lock: conv = self.engine.create_conversation(messages=req['messages'] or None)
                    _send_json(conn, {'ok': True})
                elif op == 'send':
                    if conv is None or not isinstance(req.get('msg'), str): raise ValueError('invalid message')
                    limit = req.get('max_output_tokens')
                    if limit is not None and (not isinstance(limit, int) or limit < 1): raise ValueError('invalid token limit')
                    with self.lock: res = conv.send_message(req['msg'], max_output_tokens=limit)
                    _send_json(conn, {'response': res})
                else:
                    _send_json(conn, {'ok': True}); break
        except (EOFError, OSError): pass
        except Exception as e:
            try: _send_json(conn, {'error': f'{type(e).__name__}: {e}'})
            except OSError: pass
        finally:
            if conv is not None:
                with self.lock: conv.close()
            conn.close()
            with self.state_lock:
                self.clients.discard(conn)
                self.handlers.discard(threading.current_thread())

    def close(self):
        with self.state_lock:
            listener, thread, handlers = self.listener, self.thread, list(self.handlers)
            self.stopping, self.listener, self.thread = True, None, None
            clients = list(self.clients)
        if listener is not None: listener.close()
        for conn in clients: conn.close()
        for thread in [thread, *handlers]:
            if thread is not None and thread is not threading.current_thread(): thread.join()
        with self.lock:
            if self._own_engine and hasattr(self.engine, 'close'):
                self.engine.close(); self.engine = None
        Path(self.address).unlink(missing_ok=True)

class LitertBrokerChat:
    'One isolated string conversation on a `LitertBroker` engine.'
    def __init__(self, address, messages=None):
        messages = messages or []
        if not all(isinstance(m, str) for m in messages): raise TypeError('broker messages must be strings')
        self.conn, self.lock = socket.socket(socket.AF_UNIX), threading.Lock()
        try:
            self.conn.connect(str(address)); self._call({'op': 'open', 'messages': messages})
        except Exception:
            self.conn.close(); raise

    def _call(self, req):
        _send_json(self.conn, req); res = _recv_json(self.conn)
        if 'error' in res: raise RuntimeError(res['error'])
        return res

    def __call__(self, msg, max_output_tokens=None):
        if not isinstance(msg, str): raise TypeError('broker messages must be strings')
        with self.lock: return Resp(self._call({'op': 'send', 'msg': msg, 'max_output_tokens': max_output_tokens})['response'])

    def close(self):
        with self.lock:
            if getattr(self, 'conn', None) is None: return
            try: self._call({'op': 'close'})
            except (OSError, EOFError): pass
            self.conn.close(); self.conn = None


In [ ]:
#| hide
from tempfile import TemporaryDirectory

class _BrokerConv:
    def __init__(self, n): self.n, self.messages, self.closed = n, [], False
    def send_message(self, msg, max_output_tokens=None):
        self.messages.append(msg)
        return {'role': 'assistant', 'content': [{'type': 'text', 'text': f'{self.n}:{msg}'}]}
    def close(self): self.closed = True

class _BrokerEngine:
    def __init__(self): self.convs, self.closed = [], False
    def create_conversation(self, messages=None):
        conv = _BrokerConv(len(self.convs)); self.convs.append(conv); return conv
    def close(self): self.closed = True

with TemporaryDirectory() as tmp:
    address, engine = Path(tmp) / 'litert.sock', _BrokerEngine()
    broker = LitertBroker(address, engine).start()
    first, second = LitertBrokerChat(address), LitertBrokerChat(address)
    test_eq(resp_text(first('one')), '0:one')
    test_eq(resp_text(second('two')), '1:two')
    test_eq([c.messages for c in engine.convs], [['one'], ['two']])
    first.close(); second.close(); broker.close()
    assert all(c.closed for c in engine.convs) and not engine.closed
    broker.start(); third = LitertBrokerChat(address)
    test_eq(resp_text(third('three')), '2:three')
    third.close(); broker.close()

with TemporaryDirectory() as tmp:
    address, occupied = Path(tmp) / 'litert.sock', socket.socket(socket.AF_UNIX)
    occupied.bind(str(address)); occupied.listen()
    try: LitertBroker(address, _BrokerEngine()).start()
    except RuntimeError: pass
    else: raise AssertionError('active broker socket should not be replaced')
    occupied.close()

print('litert broker tests passed')

In [ ]:
#| hide
from rishi.core import ChatBroker

assert ChatBroker is not None
